# PDF -> Per-Question Screenshots

Notebook version of `extract_exam_questions.py`.

Split a CertyIQ-style exam-dump PDF into **one PNG image per question** (including
the answer box). Questions that span multiple pages are stitched vertically into a
single tall image.

**Workflow:** drop the exam PDF into `RawData/PDF/`, then run all cells. The most
recently modified `*.pdf` is used automatically (or pin one with `PDF_FILENAME`).
Output PNGs (`Question_01.png`, ...) land in `RawData/scratch/question_shots/`.

Requires `pymupdf` and `pillow` (install cell below).

In [ ]:
# Run once if these are not already installed:
# %pip install pymupdf pillow

import os
import re
import zipfile
from pathlib import Path

import fitz  # PyMuPDF
from PIL import Image

## Configuration

Everything you might tune lives here.

In [ ]:
# --- What to process --------------------------------------------------------
# Pin a specific file in RawData/PDF. If None, the most recently modified *.pdf
# is used (so dropping in a new PDF "just works").
PDF_FILENAME: str | None = None

# --- Rendering / detection knobs -------------------------------------------
DPI = 150                          # render resolution; use 200 for sharper text
PATTERN = r"^Question:\s*(\d+)\b"  # regex for the question header; group 1 = number
MAX_HEADER_LEN = 40                # max length of a header line (filters body-text matches)
SKIP_LAST_PAGES = 1                # trailing pages to exclude (e.g. a thank-you page)
PAD_PTS = 8.0                      # padding in points above each header
MAKE_ZIP = False                   # also bundle the PNGs into <out>.zip

# --- Paths ------------------------------------------------------------------
def find_project_root(start: Path) -> Path:
    """Walk up from `start` until a folder containing 'RawData' is found."""
    for candidate in (start, *start.parents):
        if (candidate / "RawData").is_dir():
            return candidate
    raise FileNotFoundError(
        f"Could not locate project root (a parent of {start} containing 'RawData')."
    )


PROJECT_ROOT = find_project_root(Path.cwd())
PDF_DIR = PROJECT_ROOT / "RawData" / "PDF"
OUTPUT_DIR = PROJECT_ROOT / "RawData" / "scratch" / "question_shots"

print(f"Project root : {PROJECT_ROOT}")
print(f"PDF dir      : {PDF_DIR}")
print(f"Output dir   : {OUTPUT_DIR}")

In [ ]:
# --- Locate and validate the PDF -------------------------------------------
def resolve_pdf(pdf_dir: Path, filename: str | None) -> Path:
    """Return the PDF to process, validating it exists and is non-empty.

    With no `filename`, the most recently modified *.pdf is used so that dropping
    a new file into the folder is picked up automatically.
    """
    if not pdf_dir.is_dir():
        raise FileNotFoundError(f"PDF directory does not exist: {pdf_dir}")

    if filename:
        path = pdf_dir / filename
        if not path.is_file():
            raise FileNotFoundError(f"Configured PDF not found: {path}")
    else:
        pdfs = sorted(pdf_dir.glob("*.pdf"))
        if not pdfs:
            raise FileNotFoundError(f"No '*.pdf' files found in {pdf_dir}")
        if len(pdfs) > 1:
            pdfs.sort(key=lambda p: p.stat().st_mtime, reverse=True)
            print(f"Multiple PDFs found; using the most recently modified: {pdfs[0].name}")
        path = pdfs[0]

    if path.stat().st_size == 0:
        raise ValueError(f"PDF is empty: {path}")
    return path


PDF_PATH = resolve_pdf(PDF_DIR, PDF_FILENAME)
print(f"Using PDF: {PDF_PATH.name}  ({PDF_PATH.stat().st_size / 1_048_576:.2f} MB)")

## Functions

`find_question_markers` locates each question header; `extract` renders and crops
each question's page span into a stitched PNG.

In [ ]:
def find_question_markers(doc, pattern, max_header_len):
    """Locate the (page, y-position, question-number) of each question header.

    A 'header' is a text block whose first line matches the pattern and is
    short (real headers are just 'Question: N', while case-study body text
    that repeats the phrase tends to be part of a long paragraph block).
    """
    pat = re.compile(pattern)
    markers = []
    for pno in range(len(doc)):
        for block in doc[pno].get_text("blocks"):
            x0, y0, x1, y1, text, *_ = block
            first_line = text.strip().splitlines()[0] if text.strip() else ""
            m = pat.match(first_line)
            if m and len(first_line) <= max_header_len:
                markers.append((pno, y0, int(m.group(1))))

    # Keep only the first occurrence of each question number, in reading order
    markers.sort(key=lambda t: (t[0], t[1]))
    seen, clean = set(), []
    for pno, y, q in markers:
        if q not in seen:
            seen.add(q)
            clean.append((pno, y, q))
    return clean

In [ ]:
def extract(pdf_path, out_dir, dpi, pattern, max_header_len, skip_last_pages, pad_pts):
    doc = fitz.open(pdf_path)
    os.makedirs(out_dir, exist_ok=True)

    markers = find_question_markers(doc, pattern, max_header_len)
    if not markers:
        raise RuntimeError(
            "No question headers found. Check PATTERN "
            "(default expects lines like 'Question: 12')."
        )
    print(f"Detected {len(markers)} questions: {markers[0][2]}..{markers[-1][2]}")

    zoom = dpi / 72.0
    mat = fitz.Matrix(zoom, zoom)
    last_content_page = len(doc) - 1 - skip_last_pages

    page_cache = {}

    def render(pno):
        if pno not in page_cache:
            pix = doc[pno].get_pixmap(matrix=mat)
            page_cache[pno] = Image.frombytes(
                "RGB", (pix.width, pix.height), pix.samples
            )
        return page_cache[pno]

    saved = []
    for i, (spno, sy, q) in enumerate(markers):
        # A question ends where the next question header begins,
        # or at the bottom of the last content page.
        if i + 1 < len(markers):
            epno, ey, _ = markers[i + 1]
        else:
            epno, ey = last_content_page, doc[last_content_page].rect.height

        parts = []
        for pno in range(spno, epno + 1):
            img = render(pno)
            top = int(max(0, sy - pad_pts) * zoom) if pno == spno else 0
            bot = int(max(0, ey - pad_pts) * zoom) if pno == epno else img.height
            bot = min(bot, img.height)
            if bot - top < 20:  # skip slivers
                continue
            parts.append(img.crop((0, top, img.width, bot)))

        if not parts:
            print(f"  Q{q}: skipped (empty segment)")
            continue

        width = max(p.width for p in parts)
        height = sum(p.height for p in parts)
        canvas = Image.new("RGB", (width, height), "white")
        y = 0
        for p in parts:
            canvas.paste(p, (0, y))
            y += p.height

        out_path = os.path.join(out_dir, f"Question_{q:02d}.png")
        canvas.save(out_path, optimize=True)
        saved.append(out_path)
        span = f"p{spno + 1}" if spno == epno else f"p{spno + 1}-{epno + 1}"
        print(f"  Q{q}: {span}  ->  {os.path.basename(out_path)}")

        # Free rendered pages we no longer need
        for pno in list(page_cache):
            if pno < spno:
                del page_cache[pno]

    return saved

## Run

Render every question to a PNG in the output folder.

In [ ]:
saved = extract(
    str(PDF_PATH),
    str(OUTPUT_DIR),
    DPI,
    PATTERN,
    MAX_HEADER_LEN,
    SKIP_LAST_PAGES,
    PAD_PTS,
)

print(f"\nDone. {len(saved)} question images in '{OUTPUT_DIR}'.")

In [ ]:
# Optional: bundle the PNGs into a single .zip next to the output folder.
if MAKE_ZIP and saved:
    zip_path = str(OUTPUT_DIR) + ".zip"
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
        for f in saved:
            zf.write(f, os.path.basename(f))
    print(f"Zipped {len(saved)} images -> {zip_path}")

## Preview

Show the first extracted question inline (scaled down) as a sanity check.

In [ ]:
from IPython.display import display

if saved:
    preview = Image.open(saved[0])
    preview.thumbnail((700, 2000))  # scale down for display only
    print(os.path.basename(saved[0]))
    display(preview)
else:
    print("Nothing was extracted -- adjust PATTERN / MAX_HEADER_LEN and re-run.")